# 🎙️ Amazon Transcribe Lab
## Enhance Transcription Accuracy and Privacy with Custom Vocabulary & PII Redaction

---

### 📋 What We'll Do:
1. ✅ Create an S3 bucket & upload a sample audio file
2. ✅ Run a **Baseline** transcription job
3. ✅ Create a **Custom Vocabulary** and run a custom transcription job
4. ✅ Run a **PII-Redacted** transcription job
5. ✅ Compare outputs
6. ✅ **Cleanup** all AWS resources

> ⚠️ **Pre-requisite**: This notebook must run on AWS JupyterLab (SageMaker) with an IAM Role that has permissions for **S3** and **Amazon Transcribe**.

---

## 📦 Cell 1 — Install & Import Dependencies

In [ ]:
# Install required libraries
!pip install boto3 requests tqdm --quiet

import boto3
import requests
import json
import time
import uuid
from tqdm import tqdm
from datetime import datetime

print("✅ All libraries imported successfully!")

## ⚙️ Cell 2 — Configuration (Edit These Values)

In [ ]:
# ─────────────────────────────────────────────
# 🔧 CONFIGURATION — Edit if needed
# ─────────────────────────────────────────────

# AWS Region
REGION = "us-east-1"  # Change if your SageMaker is in a different region

# Unique suffix to avoid naming conflicts
UNIQUE_ID = str(uuid.uuid4())[:6]

# Resource names
BUCKET_NAME       = f"k21-transcribe-lab-{UNIQUE_ID}"
AUDIO_FILE_NAME   = "Sample_audio.wav"
BASELINE_JOB      = f"k21-baseline-job-{UNIQUE_ID}"
CUSTOM_VOCAB_NAME = f"k21-vocab-{UNIQUE_ID}"
CUSTOM_JOB        = f"k21-custom-job-{UNIQUE_ID}"
PII_JOB           = f"k21-pii-job-{UNIQUE_ID}"

# Sample audio file URL (GitHub)
AUDIO_URL = "https://github.com/k21academyuk/AWS-GenAI_AIML/raw/main/Sample_audio.wav"

print(f"""📋 Configuration Summary:
  Region          : {REGION}
  S3 Bucket       : {BUCKET_NAME}
  Audio File      : {AUDIO_FILE_NAME}
  Baseline Job    : {BASELINE_JOB}
  Custom Vocab    : {CUSTOM_VOCAB_NAME}
  Custom Job      : {CUSTOM_JOB}
  PII Job         : {PII_JOB}
""")

## 🔗 Cell 3 — Initialize AWS Clients

In [ ]:
# Initialize AWS boto3 clients using attached IAM Role (SageMaker auto-provides credentials)
s3_client          = boto3.client("s3",          region_name=REGION)
transcribe_client  = boto3.client("transcribe",  region_name=REGION)

# Verify identity
sts = boto3.client("sts", region_name=REGION)
identity = sts.get_caller_identity()

print(f"✅ Connected to AWS!")
print(f"   Account ID : {identity['Account']}")
print(f"   User/Role  : {identity['Arn']}")
print(f"   Region     : {REGION}")

## 🪣 Cell 4 — Create S3 Bucket

In [ ]:
def create_s3_bucket(bucket_name, region):
    """Create an S3 bucket in the specified region."""
    try:
        if region == "us-east-1":
            # us-east-1 does NOT accept LocationConstraint
            s3_client.create_bucket(Bucket=bucket_name)
        else:
            s3_client.create_bucket(
                Bucket=bucket_name,
                CreateBucketConfiguration={"LocationConstraint": region}
            )
        print(f"✅ S3 Bucket created: s3://{bucket_name}")
    except s3_client.exceptions.BucketAlreadyOwnedByYou:
        print(f"ℹ️  Bucket already exists and is owned by you: {bucket_name}")
    except Exception as e:
        print(f"❌ Error creating bucket: {e}")
        raise

create_s3_bucket(BUCKET_NAME, REGION)

## 🎵 Cell 5 — Download & Upload Audio File to S3

In [ ]:
def download_and_upload_audio(audio_url, audio_file_name, bucket_name):
    """Download audio file from URL and upload to S3 bucket."""
    print(f"⬇️  Downloading audio from:\n   {audio_url}")
    
    # Download the audio file
    response = requests.get(audio_url, stream=True)
    response.raise_for_status()
    
    local_path = f"/tmp/{audio_file_name}"
    with open(local_path, "wb") as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
    
    print(f"✅ Audio downloaded to: {local_path}")
    
    # Upload to S3
    print(f"⬆️  Uploading to S3: s3://{bucket_name}/{audio_file_name}")
    s3_client.upload_file(local_path, bucket_name, audio_file_name)
    
    s3_uri = f"s3://{bucket_name}/{audio_file_name}"
    print(f"✅ Audio uploaded successfully!")
    print(f"   S3 URI: {s3_uri}")
    return s3_uri

AUDIO_S3_URI = download_and_upload_audio(AUDIO_URL, AUDIO_FILE_NAME, BUCKET_NAME)

## 🔄 Cell 6 — Helper: Wait for Transcription Job

In [ ]:
def wait_for_transcription_job(job_name, poll_interval=10):
    """
    Poll Amazon Transcribe until the job is COMPLETED or FAILED.
    Returns the final job response.
    """
    print(f"⏳ Waiting for job '{job_name}' to complete...", end="")
    while True:
        response = transcribe_client.get_transcription_job(TranscriptionJobName=job_name)
        status   = response["TranscriptionJob"]["TranscriptionJobStatus"]
        
        if status == "COMPLETED":
            print(f" ✅ COMPLETED!")
            return response["TranscriptionJob"]
        elif status == "FAILED":
            reason = response["TranscriptionJob"].get("FailureReason", "Unknown reason")
            print(f" ❌ FAILED!")
            raise RuntimeError(f"Transcription job '{job_name}' failed: {reason}")
        else:
            print(".", end="", flush=True)
            time.sleep(poll_interval)


def parse_s3_uri(s3_uri):
    """
    Robustly parse any S3 URI/URL into (bucket, key).

    Handles all three formats Transcribe may return:
      1. s3://bucket/key
      2. https://s3.amazonaws.com/bucket/key          (path-style, global)
      3. https://s3.us-east-1.amazonaws.com/bucket/key (path-style, regional)
      4. https://bucket.s3.amazonaws.com/key           (virtual-hosted)
      5. https://bucket.s3.us-east-1.amazonaws.com/key (virtual-hosted, regional)
    """
    from urllib.parse import urlparse

    # Format 1: native S3 URI
    if s3_uri.startswith("s3://"):
        without_scheme = s3_uri[5:]
        bucket, key = without_scheme.split("/", 1)
        return bucket, key

    parsed  = urlparse(s3_uri)
    host    = parsed.netloc   # e.g. 's3.us-east-1.amazonaws.com' or 'bucket.s3.amazonaws.com'
    path    = parsed.path.lstrip("/")

    # Format 2 & 3: path-style  →  host starts with 's3.' or is exactly 's3.amazonaws.com'
    # netloc examples: 's3.amazonaws.com', 's3.us-east-1.amazonaws.com'
    host_parts = host.split(".")
    if host_parts[0] == "s3":
        # path = 'bucket-name/key/to/file'
        bucket, key = path.split("/", 1)
        return bucket, key

    # Format 4 & 5: virtual-hosted  →  host = 'bucket.s3[.region].amazonaws.com'
    # The bucket is everything before the first '.s3'
    if ".s3." in host or host.endswith(".s3.amazonaws.com"):
        bucket = host.split(".s3.")[0]
        key    = path
        return bucket, key

    raise ValueError(f"Cannot parse S3 URI — unrecognised format: {s3_uri}")


def read_s3_json(uri):
    """
    Read a JSON file from S3 using boto3 (works for private objects).
    """
    bucket, key = parse_s3_uri(uri)
    print(f"   📥 Reading s3://{bucket}/{key}")
    obj  = s3_client.get_object(Bucket=bucket, Key=key)
    body = obj["Body"].read().decode("utf-8")
    return json.loads(body)


def fetch_transcript_text(job_result, label="Transcript"):
    """
    Download and return transcript text from the Transcribe output URI.
    Uses boto3 S3 (not requests.get) so private S3 objects work correctly.
    For PII jobs, fetches both redacted and unredacted transcripts.
    """
    transcripts = {}

    # Standard / unredacted transcript
    uri = job_result.get("Transcript", {}).get("TranscriptFileUri")
    if uri:
        data = read_s3_json(uri)
        text = data["results"]["transcripts"][0]["transcript"]
        transcripts[label] = text

    # Redacted transcript (PII jobs only)
    redacted_uri = job_result.get("Transcript", {}).get("RedactedTranscriptFileUri")
    if redacted_uri:
        data = read_s3_json(redacted_uri)
        text = data["results"]["transcripts"][0]["transcript"]
        transcripts[f"{label} (Redacted)"] = text

    return transcripts

print("✅ Helper functions defined.")

## 🏁 Cell 7 — Step 1: Baseline Transcription Job

In [ ]:
print("="*60)
print("🏁 STEP 1: BASELINE TRANSCRIPTION JOB")
print("="*60)
print("No custom vocabulary. No PII redaction. Pure baseline.\n")

transcribe_client.start_transcription_job(
    TranscriptionJobName = BASELINE_JOB,
    Media                = {"MediaFileUri": AUDIO_S3_URI},
    MediaFormat          = "wav",
    LanguageCode         = "en-US",
    OutputBucketName     = BUCKET_NAME,
    OutputKey            = f"transcripts/{BASELINE_JOB}.json"
)

print(f"🚀 Baseline job started: {BASELINE_JOB}")

baseline_job_result = wait_for_transcription_job(BASELINE_JOB)
baseline_transcripts = fetch_transcript_text(baseline_job_result, label="Baseline")

print("\n📄 BASELINE TRANSCRIPT:")
print("-"*60)
for label, text in baseline_transcripts.items():
    print(f"[{label}]:\n{text}\n")

## 📚 Cell 8 — Step 2: Create Custom Vocabulary

In [ ]:
print("="*60)
print("📚 STEP 2: CREATE CUSTOM VOCABULARY")
print("="*60)

# Custom vocabulary entries
# Format: Phrase (required), IPA (optional), SoundsLike (optional), DisplayAs (optional)
vocab_phrases = [
    {"Phrase": "lotus-elise",      "SoundsLike": "lo-tus-e-lise",      "DisplayAs": "Lotus Elise"},
    {"Phrase": "turbocharger",     "SoundsLike": "tur-bo-char-ger",     "DisplayAs": "Turbocharger"},
    {"Phrase": "horsepower",       "SoundsLike": "horse-pow-er",        "DisplayAs": "Horsepower"},
    {"Phrase": "aerodynamics",     "SoundsLike": "air-o-dy-nam-ics",    "DisplayAs": "Aerodynamics"},
    {"Phrase": "transmission",     "SoundsLike": "trans-mis-sion",      "DisplayAs": "Transmission"},
]

print("📝 Custom Vocabulary Entries:")
for entry in vocab_phrases:
    print(f"   Phrase: {entry['Phrase']:25s} → DisplayAs: {entry.get('DisplayAs', '')}")

print(f"\n🚀 Creating vocabulary: {CUSTOM_VOCAB_NAME}")

transcribe_client.create_vocabulary(
    VocabularyName = CUSTOM_VOCAB_NAME,
    LanguageCode   = "en-US",
    Phrases        = [p["Phrase"] for p in vocab_phrases]
)

# Wait for vocabulary to be READY
print("⏳ Waiting for vocabulary to be ready...", end="")
while True:
    vocab_resp = transcribe_client.get_vocabulary(VocabularyName=CUSTOM_VOCAB_NAME)
    vocab_state = vocab_resp["VocabularyState"]
    if vocab_state == "READY":
        print(" ✅ READY!")
        break
    elif vocab_state == "FAILED":
        print(" ❌ FAILED!")
        raise RuntimeError(f"Vocabulary creation failed: {vocab_resp.get('FailureReason')}")
    else:
        print(".", end="", flush=True)
        time.sleep(5)

print(f"\n✅ Custom Vocabulary '{CUSTOM_VOCAB_NAME}' is ready!")

## 🎯 Cell 9 — Step 3: Custom Vocabulary Transcription Job

In [ ]:
print("="*60)
print("🎯 STEP 3: CUSTOM VOCABULARY TRANSCRIPTION JOB")
print("="*60)
print("Same audio, but now using our custom vocabulary.\n")

transcribe_client.start_transcription_job(
    TranscriptionJobName = CUSTOM_JOB,
    Media                = {"MediaFileUri": AUDIO_S3_URI},
    MediaFormat          = "wav",
    LanguageCode         = "en-US",
    OutputBucketName     = BUCKET_NAME,
    OutputKey            = f"transcripts/{CUSTOM_JOB}.json",
    Settings             = {
        "VocabularyName": CUSTOM_VOCAB_NAME
    }
)

print(f"🚀 Custom vocabulary job started: {CUSTOM_JOB}")

custom_job_result = wait_for_transcription_job(CUSTOM_JOB)
custom_transcripts = fetch_transcript_text(custom_job_result, label="Custom Vocab")

print("\n📄 CUSTOM VOCABULARY TRANSCRIPT:")
print("-"*60)
for label, text in custom_transcripts.items():
    print(f"[{label}]:\n{text}\n")

## 🔐 Cell 10 — Step 4: PII-Redacted Transcription Job

In [ ]:
print("="*60)
print("🔐 STEP 4: PII-REDACTED TRANSCRIPTION JOB")
print("="*60)
print("PII entities like names, addresses, phone numbers will")
print("be replaced with [PII] tags in the redacted output.\n")

transcribe_client.start_transcription_job(
    TranscriptionJobName = PII_JOB,
    Media                = {"MediaFileUri": AUDIO_S3_URI},
    MediaFormat          = "wav",
    LanguageCode         = "en-US",
    OutputBucketName     = BUCKET_NAME,
    OutputKey            = f"transcripts/{PII_JOB}.json",
    ContentRedaction     = {
        "RedactionType"   : "PII",
        "RedactionOutput" : "redacted_and_unredacted",   # Produces BOTH versions
        "PiiEntityTypes"  : [
            "NAME", "ADDRESS", "PHONE", "EMAIL",
            "SSN", "CREDIT_DEBIT_NUMBER", "DATE_TIME",
            "AGE", "PIN", "BANK_ACCOUNT_NUMBER"
        ]
    }
)

print(f"🚀 PII redaction job started: {PII_JOB}")

pii_job_result = wait_for_transcription_job(PII_JOB)
pii_transcripts = fetch_transcript_text(pii_job_result, label="PII Job")

print("\n📄 PII TRANSCRIPTION RESULTS:")
print("-"*60)
for label, text in pii_transcripts.items():
    print(f"[{label}]:\n{text}\n")

## 📊 Cell 11 — Compare All Three Transcripts Side by Side

In [ ]:
print("="*70)
print("📊 FINAL COMPARISON — ALL THREE TRANSCRIPTION APPROACHES")
print("="*70)

all_results = {
    **baseline_transcripts,
    **custom_transcripts,
    **pii_transcripts
}

for i, (label, text) in enumerate(all_results.items(), 1):
    print(f"\n{'─'*70}")
    print(f"  [{i}] {label.upper()}")
    print(f"{'─'*70}")
    print(text)

print(f"\n{'='*70}")
print("✅ All transcription jobs completed and compared!")
print(f"{'='*70}")
print("""
📌 KEY OBSERVATIONS:
  1. Baseline     → Raw transcription, no customization.
  2. Custom Vocab → Domain-specific terms recognized correctly.
  3. PII Job      → Sensitive info replaced with [PII] tags.
  4. Unredacted   → Full transcript stored separately for audit.
""")

## 📁 Cell 12 — View All Output Files in S3

In [ ]:
print("📁 Files stored in your S3 bucket:")
print(f"   s3://{BUCKET_NAME}/\n")

paginator = s3_client.get_paginator("list_objects_v2")
pages     = paginator.paginate(Bucket=BUCKET_NAME)

total_files = 0
for page in pages:
    for obj in page.get("Contents", []):
        size_kb = obj["Size"] / 1024
        print(f"   📄 {obj['Key']:<60s} ({size_kb:.1f} KB)")
        total_files += 1

print(f"\n✅ Total files in bucket: {total_files}")

---
## 🧹 Cell 13 — CLEANUP: Delete All AWS Resources

> ⚠️ **This cell deletes everything created in this lab.**  
> Run this when you're done to avoid unnecessary AWS charges.

Resources deleted:
- All 3 Amazon Transcribe jobs
- Custom Vocabulary
- All objects in the S3 bucket
- The S3 bucket itself

In [ ]:
print("="*60)
print("🧹 CLEANUP — DELETING ALL LAB RESOURCES")
print("="*60)

errors = []

# ── 1. Delete Transcription Jobs ──────────────────────────────
print("\n🗑️  Deleting Amazon Transcribe jobs...")
for job_name in [BASELINE_JOB, CUSTOM_JOB, PII_JOB]:
    try:
        transcribe_client.delete_transcription_job(TranscriptionJobName=job_name)
        print(f"   ✅ Deleted job: {job_name}")
    except transcribe_client.exceptions.NotFoundException:
        print(f"   ⚠️  Job not found (already deleted?): {job_name}")
    except Exception as e:
        msg = f"   ❌ Error deleting job {job_name}: {e}"
        print(msg)
        errors.append(msg)

# ── 2. Delete Custom Vocabulary ───────────────────────────────
print("\n🗑️  Deleting Custom Vocabulary...")
try:
    transcribe_client.delete_vocabulary(VocabularyName=CUSTOM_VOCAB_NAME)
    print(f"   ✅ Deleted vocabulary: {CUSTOM_VOCAB_NAME}")
except transcribe_client.exceptions.NotFoundException:
    print(f"   ⚠️  Vocabulary not found (already deleted?): {CUSTOM_VOCAB_NAME}")
except Exception as e:
    msg = f"   ❌ Error deleting vocabulary: {e}"
    print(msg)
    errors.append(msg)

# ── 3. Empty & Delete S3 Bucket ───────────────────────────────
print(f"\n🗑️  Emptying S3 bucket: {BUCKET_NAME}...")
try:
    # Delete all objects
    paginator = s3_client.get_paginator("list_objects_v2")
    pages     = paginator.paginate(Bucket=BUCKET_NAME)
    
    deleted_count = 0
    for page in pages:
        objects = [{"Key": obj["Key"]} for obj in page.get("Contents", [])]
        if objects:
            s3_client.delete_objects(
                Bucket=BUCKET_NAME,
                Delete={"Objects": objects}
            )
            deleted_count += len(objects)
    
    print(f"   ✅ Deleted {deleted_count} object(s) from bucket.")
    
    # Now delete the bucket itself
    s3_client.delete_bucket(Bucket=BUCKET_NAME)
    print(f"   ✅ Deleted S3 bucket: {BUCKET_NAME}")

except s3_client.exceptions.NoSuchBucket:
    print(f"   ⚠️  Bucket not found (already deleted?): {BUCKET_NAME}")
except Exception as e:
    msg = f"   ❌ Error deleting bucket: {e}"
    print(msg)
    errors.append(msg)

# ── Summary ───────────────────────────────────────────────────
print("\n" + "="*60)
if errors:
    print("⚠️  CLEANUP COMPLETED WITH SOME ERRORS:")
    for err in errors:
        print(err)
else:
    print("✅ ALL RESOURCES DELETED SUCCESSFULLY!")
    print("   No further AWS charges will be incurred for this lab.")
print("="*60)

---
## 🎉 Lab Complete!

### What You Accomplished:

| Step | Task | Status |
|------|------|--------|
| 1 | Created S3 bucket & uploaded audio | ✅ |
| 2 | Ran Baseline transcription job | ✅ |
| 3 | Created Custom Vocabulary | ✅ |
| 4 | Ran Custom Vocabulary transcription job | ✅ |
| 5 | Ran PII-Redacted transcription job | ✅ |
| 6 | Compared all outputs | ✅ |
| 7 | Cleaned up all AWS resources | ✅ |

### 💡 Key Learnings:
- **Amazon Transcribe** can convert speech to text automatically.
- **Custom Vocabulary** improves accuracy for domain-specific terms.
- **PII Redaction** protects sensitive data (names, phones, addresses) with `[PII]` tags.
- Both redacted and unredacted versions can be stored for compliance and audit.

---
> 📢 Share your experience on LinkedIn and tag **K21Academy** & **Atul Kumar**!